In [2]:
import pandas as pd
import pyarrow.parquet as pq
import pickle
import os

# ----------------------------
# Config
# ----------------------------
FLOW_FILE = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples\DrDoS_DNS_5k_samples.parquet"
PACKET_DIR = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\01-12\packet-level\packets"
MAPPING_FILE = r"/models/model_main/data/mapping/packet_level_mapping/packet_file_flow_mapping_no_duplicates.pkl"
OUTPUT_FILE = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDos_Dns_100_packets_per_flow.parquet"

CHUNK_SIZE = 30_000
MAX_PACKETS_PER_FLOW = 100

# ----------------------------
# Load flow IDs
# ----------------------------
flows = pd.read_parquet(FLOW_FILE, columns=["Flow ID"])
flow_ids = flows["Flow ID"].tolist()
print(f"Total flows to process: {len(flow_ids)}")

# ----------------------------
# Load mapping file
# ----------------------------
with open(MAPPING_FILE, "rb") as f:
    flow_to_file = pickle.load(f)

# ----------------------------
# Prepare result storage
# ----------------------------
all_flows = []

# ----------------------------
# Process flows
# ----------------------------
for i, flow_id in enumerate(flow_ids, start=1):
    packet_file_name = flow_to_file.get(flow_id)
    if not packet_file_name:
        continue  # skip if no mapping

    packet_file_path = os.path.join(PACKET_DIR, packet_file_name)
    if not os.path.exists(packet_file_path):
        continue  # skip if file missing

    # Open Parquet file with PyArrow
    parquet_file = pq.ParquetFile(packet_file_path)
    flow_packets = pd.DataFrame()

    # Iterate over batches/chunks
    for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
        df_chunk = batch.to_pandas()
        df_flow = df_chunk[df_chunk["flow_id"] == flow_id]
        if not df_flow.empty:
            flow_packets = pd.concat([flow_packets, df_flow], ignore_index=True)
            # Keep only earliest MAX_PACKETS_PER_FLOW
            flow_packets = flow_packets.sort_values("timestamp").head(MAX_PACKETS_PER_FLOW)

        if len(flow_packets) >= MAX_PACKETS_PER_FLOW:
            break  # we already have 100 packets

    if not flow_packets.empty:
        flow_packets = flow_packets.sort_values("timestamp").head(MAX_PACKETS_PER_FLOW)
        all_flows.append(flow_packets)

    if i % 100 == 0:
        print(f"Processed {i}/{len(flow_ids)} flows...")

# ----------------------------
# Combine all flows and save
# ----------------------------
final_df = pd.concat(all_flows, ignore_index=True)
final_df.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved all flows to {OUTPUT_FILE}")


Total flows to process: 5000
Processed 100/5000 flows...
Processed 200/5000 flows...
Processed 300/5000 flows...
Processed 400/5000 flows...
Processed 500/5000 flows...
Processed 600/5000 flows...
Processed 700/5000 flows...
Processed 800/5000 flows...
Processed 900/5000 flows...
Processed 1000/5000 flows...
Processed 1100/5000 flows...
Processed 1200/5000 flows...
Processed 1300/5000 flows...
Processed 1400/5000 flows...
Processed 1500/5000 flows...
Processed 1600/5000 flows...
Processed 1700/5000 flows...
Processed 1800/5000 flows...
Processed 1900/5000 flows...
Processed 2000/5000 flows...
Processed 2100/5000 flows...
Processed 2200/5000 flows...
Processed 2300/5000 flows...
Processed 2400/5000 flows...
Processed 2500/5000 flows...
Processed 2600/5000 flows...
Processed 2700/5000 flows...
Processed 2800/5000 flows...
Processed 2900/5000 flows...
Processed 3000/5000 flows...
Processed 3100/5000 flows...
Processed 3200/5000 flows...
Processed 3300/5000 flows...
Processed 3400/5000 flo

In [4]:
path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDos_Dns_100_packets_per_flow.parquet"

df = pd.read_parquet(path)
print(f"Total rows in final DataFrame: {len(df)}")
print(f"Columns in final DataFrame: {df.columns.tolist()}")
print(f"Sample data:\n{df.head()}")
print(f"Unique flow IDs: {df['flow_id'].nunique()}")
print(f"Total packets per flow: {df.groupby('flow_id').size().max()}")
print(f"Total unique flows: {df['flow_id'].nunique()}")

Total rows in final DataFrame: 197238
Columns in final DataFrame: ['timestamp', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'length', 'ttl', 'payload_size', 'ip_id', 'ip_flags', 'ip_fragment_offset', 'ip_tos', 'tcp_flags', 'tcp_window', 'tcp_seq', 'tcp_ack', 'tcp_urgent', 'icmp_type', 'icmp_code', 'flow_id', 'packet_position', 'inter_arrival_time', 'payload_entropy', 'packet_direction']
Sample data:
      timestamp      src_ip        dst_ip  src_port  dst_port  protocol  \
0  1.543675e+09  172.16.0.5  192.168.50.1     634.0   60495.0        17   
1  1.543675e+09  172.16.0.5  192.168.50.1     634.0   60495.0        17   
2  1.543675e+09  172.16.0.5  192.168.50.1     634.0   60495.0        17   
3  1.543675e+09  172.16.0.5  192.168.50.1     634.0   60495.0        17   
4  1.543675e+09  172.16.0.5  192.168.50.1     634.0   60495.0        17   

   length  ttl  payload_size  ip_id  ...  tcp_seq  tcp_ack  tcp_urgent  \
0     482   47           440  34451  ...      NaN      NaN  

In [1]:
import os
import pandas as pd
import pickle
import pyarrow.parquet as pq
from collections import defaultdict
import gc

def process_flows_correctly():
    # Paths
    flow_file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples\DrDoS_MSSQL_5k_samples.parquet"
    packet_folder = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\data\01-12\packet-level\packets"
    mapping_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\mapping\packet_level_mapping\packet_file_flow_mapping.pkl"
    output_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDoS_MSSQL_100_packets_per_flow.parquet"

    # Create output directory
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    print("Loading flow IDs...")
    # Read only Flow ID column from flow-level data
    flow_df = pd.read_parquet(flow_file_path, columns=['Flow ID'])  # UPDATE COLUMN NAME IF NEEDED
    target_flow_ids = set(flow_df['Flow ID'].unique())
    print(f"Found {len(target_flow_ids)} target flow IDs")
    del flow_df
    gc.collect()

    print("Loading mapping file...")
    # Load the mapping file
    with open(mapping_path, 'rb') as f:
        file_to_flows = pickle.load(f)

    # Create reverse mapping: flow_id -> list of files containing it
    print("Creating reverse mapping...")
    flow_to_files = defaultdict(list)
    for file_name, flow_ids in file_to_flows.items():
        for flow_id in flow_ids:
            if flow_id in target_flow_ids:
                flow_to_files[flow_id].append(file_name)

    print(f"Flow-to-files mapping created for {len(flow_to_files)} flows")
    del file_to_flows
    gc.collect()

    # Process flows in small batches to manage memory
    batch_size = 50  # Smaller batches for better memory control
    flow_ids_list = list(flow_to_files.keys())
    total_flows = len(flow_ids_list)

    # List to store results in chunks
    result_chunks = []

    for batch_start in range(0, total_flows, batch_size):
        batch_end = min(batch_start + batch_size, total_flows)
        batch_flows = flow_ids_list[batch_start:batch_end]

        print(f"Processing flows {batch_start+1}-{batch_end} of {total_flows}")

        # Get all unique files needed for this batch
        files_needed = set()
        for flow_id in batch_flows:
            files_needed.update(flow_to_files[flow_id])

        print(f"  Need to read {len(files_needed)} packet files for this batch")

        # Store packets for current batch - use dict to accumulate per flow
        batch_flow_packets = {flow_id: [] for flow_id in batch_flows}

        # Read each needed file once and extract relevant packets
        for file_name in files_needed:
            file_path = os.path.join(packet_folder, file_name)

            try:
                # Read packet file
                packet_df = pd.read_parquet(file_path)

                # Filter for flows in current batch that are in this file
                relevant_flows = [f for f in batch_flows if f in packet_df['flow_id'].values]

                if relevant_flows:
                    # Process each flow separately
                    for flow_id in relevant_flows:
                        flow_packets = packet_df[packet_df['flow_id'] == flow_id].copy()

                        if not flow_packets.empty:
                            # Sort by timestamp FIRST
                            flow_packets = flow_packets.sort_values('timestamp')

                            # Add to accumulated packets for this flow
                            batch_flow_packets[flow_id].append(flow_packets)

                del packet_df
                gc.collect()

            except Exception as e:
                print(f"  Error reading {file_name}: {e}")
                continue

        # Process accumulated packets for each flow in batch
        batch_results = []
        for flow_id in batch_flows:
            if batch_flow_packets[flow_id]:
                # Combine all packets for this flow
                all_flow_packets = pd.concat(batch_flow_packets[flow_id], ignore_index=True)

                # Sort by timestamp again (in case packets came from multiple files)
                all_flow_packets = all_flow_packets.sort_values('timestamp')

                # Take EXACTLY the first 100 packets (or fewer if less available)
                final_flow_packets = all_flow_packets.head(100).copy()

                # Add/rename Flow ID column
                final_flow_packets['Flow ID'] = flow_id

                batch_results.append(final_flow_packets)

                print(f"  Flow {flow_id}: {len(final_flow_packets)} packets collected")

        # Combine batch results
        if batch_results:
            batch_combined = pd.concat(batch_results, ignore_index=True)
            result_chunks.append(batch_combined)

            # Verify no flow has more than 100 packets in this batch
            flow_counts = batch_combined['Flow ID'].value_counts()
            max_packets = flow_counts.max()
            if max_packets > 100:
                print(f"  ⚠️ WARNING: Found flow with {max_packets} packets in batch!")

            print(f"  Batch complete: {len(batch_combined)} packets, max per flow: {max_packets}")

            del batch_results, batch_combined, batch_flow_packets
            gc.collect()
        else:
            print(f"  No packets found for batch {batch_start+1}-{batch_end}")

    # Combine all chunks and save
    if result_chunks:
        print("Combining all results...")
        final_result = pd.concat(result_chunks, ignore_index=True)

        # Final verification
        flow_counts = final_result['Flow ID'].value_counts()
        max_packets_final = flow_counts.max()
        flows_over_100 = (flow_counts > 100).sum()

        print(f"Final dataset: {len(final_result)} rows, {len(final_result['Flow ID'].unique())} unique flows")
        print(f"Max packets per flow: {max_packets_final}")
        print(f"Flows with >100 packets: {flows_over_100}")

        if flows_over_100 > 0:
            print("❌ ERROR: Still have flows with >100 packets!")
            print("Flows with >100 packets:")
            print(flow_counts[flow_counts > 100].head(10))
            return False
        else:
            print("✅ All flows have ≤100 packets")

        # Reorder columns to put Flow ID first
        cols = ['Flow ID'] + [col for col in final_result.columns if col != 'Flow ID' and col != 'flow_id']
        # Remove duplicate flow_id column if it exists
        if 'flow_id' in final_result.columns:
            final_result = final_result.drop('flow_id', axis=1)
        final_result = final_result[cols]

        # Save to parquet
        final_result.to_parquet(output_path, index=False)
        print(f"Results saved to: {output_path}")

        # Print sample stats
        flows_with_100 = (flow_counts == 100).sum()
        flows_with_less = (flow_counts < 100).sum()

        print(f"Flows with exactly 100 packets: {flows_with_100} ({(flows_with_100/len(flow_counts))*100:.1f}%)")
        print(f"Flows with <100 packets: {flows_with_less} ({(flows_with_less/len(flow_counts))*100:.1f}%)")

        return True

    else:
        print("No packets found for any flows!")
        return False

In [2]:
success = process_flows_correctly()
if success:
    print("\n🎉 Processing completed successfully!")
else:
    print("\n❌ Processing failed - check the logs above")

Loading flow IDs...
Found 5000 target flow IDs
Loading mapping file...
Creating reverse mapping...
Flow-to-files mapping created for 5000 flows
Processing flows 1-50 of 5000
  Need to read 83 packet files for this batch
  Flow 172.16.0.5_192.168.50.1_696_64964_17: 100 packets collected
  Flow 172.16.0.5_192.168.50.1_634_52301_17: 100 packets collected
  Flow 172.16.0.5_192.168.50.1_634_58317_17: 82 packets collected
  Flow 172.16.0.5_192.168.50.1_634_59591_17: 100 packets collected
  Flow 172.16.0.5_192.168.50.1_848_56665_17: 28 packets collected
  Flow 172.16.0.5_192.168.50.1_900_7134_17: 54 packets collected
  Flow 172.16.0.5_192.168.50.1_960_45928_17: 16 packets collected
  Flow 172.16.0.5_192.168.50.1_900_32570_17: 28 packets collected
  Flow 172.16.0.5_192.168.50.1_853_26645_17: 36 packets collected
  Flow 172.16.0.5_192.168.50.1_634_24509_17: 100 packets collected
  Flow 172.16.0.5_192.168.50.1_542_53206_17: 74 packets collected
  Flow 172.16.0.5_192.168.50.1_962_17335_17: 28 pac

In [3]:
import pandas as pd
import numpy as np

def analyze_flow_packets_file():
    file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDoS_MSSQL_100_packets_per_flow.parquet"

    print("Loading parquet file...")
    df = pd.read_parquet(file_path)

    print("="*60)
    print("BASIC FILE STATISTICS")
    print("="*60)
    print(f"Total rows: {len(df):,}")
    print(f"Total columns: {len(df.columns)}")
    print(f"File size in memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    print("\nColumns in the file:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")

    print("\n" + "="*60)
    print("FLOW ID ANALYSIS")
    print("="*60)

    # Check if Flow ID column exists
    flow_id_col = None
    possible_names = ['Flow ID', 'flow_id', 'Flow_ID', 'FlowID']
    for name in possible_names:
        if name in df.columns:
            flow_id_col = name
            break

    if flow_id_col is None:
        print("❌ No Flow ID column found!")
        print("Available columns:", list(df.columns))
        return

    print(f"✅ Flow ID column found: '{flow_id_col}'")

    # Flow statistics
    unique_flows = df[flow_id_col].unique()
    flow_counts = df[flow_id_col].value_counts()

    print(f"Total unique flows: {len(unique_flows):,}")
    print(f"Expected max rows (flows × 100): {len(unique_flows) * 100:,}")
    print(f"Actual rows: {len(df):,}")

    print("\n" + "="*60)
    print("PACKETS PER FLOW DISTRIBUTION")
    print("="*60)

    print(f"Min packets per flow: {flow_counts.min()}")
    print(f"Max packets per flow: {flow_counts.max()}")
    print(f"Mean packets per flow: {flow_counts.mean():.2f}")
    print(f"Median packets per flow: {flow_counts.median():.2f}")

    # Check if any flow has more than 100 packets
    flows_over_100 = flow_counts[flow_counts > 100]
    if len(flows_over_100) > 0:
        print(f"⚠️  WARNING: {len(flows_over_100)} flows have more than 100 packets!")
        print("Flows with >100 packets:")
        print(flows_over_100.head(10))
    else:
        print("✅ All flows have ≤100 packets")

    # Distribution of packet counts
    print(f"\nDistribution of packet counts:")
    packet_count_dist = flow_counts.value_counts().sort_index()
    for count, num_flows in packet_count_dist.items():
        percentage = (num_flows / len(unique_flows)) * 100
        print(f"  {count:3d} packets: {num_flows:4d} flows ({percentage:5.1f}%)")

    print("\n" + "="*60)
    print("TIMESTAMP ANALYSIS")
    print("="*60)

    # Check timestamp column
    timestamp_col = None
    possible_ts_names = ['timestamp', 'Timestamp', 'time', 'Time']
    for name in possible_ts_names:
        if name in df.columns:
            timestamp_col = name
            break

    if timestamp_col is None:
        print("❌ No timestamp column found!")
    else:
        print(f"✅ Timestamp column found: '{timestamp_col}'")

        # Check if timestamps are sorted within each flow
        print("Checking timestamp sorting within flows...")
        unsorted_flows = []

        # Sample a few flows to check sorting
        sample_flows = np.random.choice(unique_flows, min(10, len(unique_flows)), replace=False)

        for flow_id in sample_flows:
            flow_data = df[df[flow_id_col] == flow_id]
            timestamps = flow_data[timestamp_col].values

            if not np.all(timestamps[:-1] <= timestamps[1:]):
                unsorted_flows.append(flow_id)

        if unsorted_flows:
            print(f"⚠️  Found {len(unsorted_flows)} flows with unsorted timestamps (from sample of {len(sample_flows)})")
        else:
            print(f"✅ All sampled flows ({len(sample_flows)}) have sorted timestamps")

    print("\n" + "="*60)
    print("DATA QUALITY CHECKS")
    print("="*60)

    # Check for missing values
    print("Missing values per column:")
    missing_counts = df.isnull().sum()
    for col, missing in missing_counts.items():
        if missing > 0:
            percentage = (missing / len(df)) * 100
            print(f"  {col}: {missing:,} ({percentage:.2f}%)")

    if missing_counts.sum() == 0:
        print("✅ No missing values found")

    # Check for duplicate rows
    duplicates = df.duplicated().sum()
    print(f"\nDuplicate rows: {duplicates:,}")

    if duplicates == 0:
        print("✅ No duplicate rows found")
    else:
        print("⚠️  Found duplicate rows")

    print("\n" + "="*60)
    print("SAMPLE DATA")
    print("="*60)

    print("First 5 rows:")
    print(df.head())

    print(f"\nSample of flows and their packet counts:")
    sample_flow_counts = flow_counts.head(10)
    for flow_id, count in sample_flow_counts.items():
        print(f"  {flow_id}: {count} packets")

    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)

    expected_flows = 5000  # Your original estimate
    actual_flows = len(unique_flows)

    print(f"Expected flows: ~{expected_flows:,}")
    print(f"Actual flows: {actual_flows:,}")
    print(f"Flow coverage: {(actual_flows/expected_flows)*100:.1f}%" if expected_flows > 0 else "N/A")

    flows_with_100 = (flow_counts == 100).sum()
    flows_with_less = (flow_counts < 100).sum()

    print(f"Flows with exactly 100 packets: {flows_with_100:,} ({(flows_with_100/actual_flows)*100:.1f}%)")
    print(f"Flows with <100 packets: {flows_with_less:,} ({(flows_with_less/actual_flows)*100:.1f}%)")

    efficiency_score = len(df) / (len(unique_flows) * 100) * 100
    print(f"Data efficiency: {efficiency_score:.1f}% (actual_rows / max_possible_rows)")

if __name__ == "__main__":
    analyze_flow_packets_file()

Loading parquet file...
BASIC FILE STATISTICS
Total rows: 31,056
Total columns: 25
File size in memory: 13.65 MB

Columns in the file:
   1. Flow ID
   2. timestamp
   3. src_ip
   4. dst_ip
   5. src_port
   6. dst_port
   7. protocol
   8. length
   9. ttl
  10. payload_size
  11. ip_id
  12. ip_flags
  13. ip_fragment_offset
  14. ip_tos
  15. tcp_flags
  16. tcp_window
  17. tcp_seq
  18. tcp_ack
  19. tcp_urgent
  20. icmp_type
  21. icmp_code
  22. packet_position
  23. inter_arrival_time
  24. payload_entropy
  25. packet_direction

FLOW ID ANALYSIS
✅ Flow ID column found: 'Flow ID'
Total unique flows: 5,000
Expected max rows (flows × 100): 500,000
Actual rows: 31,056

PACKETS PER FLOW DISTRIBUTION
Min packets per flow: 2
Max packets per flow: 100
Mean packets per flow: 6.21
Median packets per flow: 2.00
✅ All flows have ≤100 packets

Distribution of packet counts:
    2 packets: 2552 flows ( 51.0%)
    4 packets: 1091 flows ( 21.8%)
    6 packets:  259 flows (  5.2%)
    8 pack

In [4]:
import pandas as pd
path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\flow_level\5k_samples\DrDoS_DNS_5k_samples.parquet"

df = pd.read_parquet(path)
df.head(5)

,Unnamed: 0,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,SimillarHTTP,Inbound,Label
0,425,172.16.0.5_192.168.50.1_634_60495_17,172.16.0.5,634,192.168.50.1,60495,17,2018-12-01 10:51:39.813448,28415,97,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
1,1654,172.16.0.5_192.168.50.1_634_46391_17,172.16.0.5,634,192.168.50.1,46391,17,2018-12-01 10:51:39.852499,48549,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
2,2927,172.16.0.5_192.168.50.1_634_11894_17,172.16.0.5,634,192.168.50.1,11894,17,2018-12-01 10:51:39.890213,48337,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
3,694,172.16.0.5_192.168.50.1_634_27878_17,172.16.0.5,634,192.168.50.1,27878,17,2018-12-01 10:51:39.941151,32026,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS
4,838,172.16.0.5_192.168.50.1_634_47149_17,172.16.0.5,634,192.168.50.1,47149,17,2018-12-01 10:51:39.942030,46469,200,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,DrDoS_DNS


## Labelling the packets based on the flow ID from the flow-level data.

In [8]:
import pandas as pd
path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDoS_MSSQL_100_packets_per_flow.parquet"

df = pd.read_parquet(path)
df['label'] = "DrDoS_MSSQL"

df.to_parquet(path, index=False)

In [9]:
path2 = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\models\model_main\data\packet_level\DrDos_MSSQL_100_packets_per_flow.parquet"

df2 = pd.read_parquet(path2)
df.head(5)

,Flow ID,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,length,ttl,payload_size,...,tcp_seq,tcp_ack,tcp_urgent,icmp_type,icmp_code,packet_position,inter_arrival_time,payload_entropy,packet_direction,label
0,172.16.0.5_192.168.50.1_696_64964_17,1.543675e+09,172.16.0.5,192.168.50.1,696.0,64964.0,17,482,50,440,...,NaN,NaN,NaN,NaN,NaN,1,0.000000e+00,3.426967,forward,DrDoS_MSSQL
1,172.16.0.5_192.168.50.1_696_64964_17,1.543675e+09,172.16.0.5,192.168.50.1,696.0,64964.0,17,482,50,440,...,NaN,NaN,NaN,NaN,NaN,2,9.536743e-07,3.426967,forward,DrDoS_MSSQL
2,172.16.0.5_192.168.50.1_696_64964_17,1.543675e+09,172.16.0.5,192.168.50.1,696.0,64964.0,17,482,50,440,...,NaN,NaN,NaN,NaN,NaN,3,1.502037e-05,3.442819,forward,DrDoS_MSSQL
3,172.16.0.5_192.168.50.1_696_64964_17,1.543675e+09,172.16.0.5,192.168.50.1,696.0,64964.0,17,482,50,440,...,NaN,NaN,NaN,NaN,NaN,4,0.000000e+00,3.442819,forward,DrDoS_MSSQL
4,172.16.0.5_192.168.50.1_696_64964_17,1.543675e+09,172.16.0.5,192.168.50.1,696.0,64964.0,17,482,50,440,...,NaN,NaN,NaN,NaN,NaN,5,9.536743e-07,3.434155,forward,DrDoS_MSSQL
